In [1]:
import akshare as ak
import pandas as pd
import numpy as np

# 学习目标:学会使用AI辅助工具，进行数据的基本分析
## ---以某只股票为例，分析收益率的计算，包括简单收益和经通货膨胀调整的收益
## ---计算波动率
## ---实现可视化
## ---可视化均线和成交量

### 1、金融产品的收益率（Return）

在金融学中，衡量金融资产在两个时期之间价格变动的基本方式是**收益率（Return）**。最常用的两种形式是：

* **简单收益率（Simple Return）**
* **对数收益率（Log Return）**

下面分别介绍。

---

## 1. 简单收益率（Simple Return）

设资产在时点 $(t-1)$ 的价格为 $p(t-1)$，在时点 $t$ 的价格为 $p(t)$，则**简单收益率**定义为：

$$
R(t) = \frac{p(t) - p(t-1)}{p(t-1)}
$$

也可写成：

$$
R(t) = \frac{p(t)}{p(t-1)} - 1
$$

解释：

* $R(t) = 0.05$ 表示价格上涨 $5%$
* $R(t) = -0.02$ 表示价格下跌 $2%$

若期间获得现金流（如股息或票息）记为 $D(t)$，则调整后的简单收益率为：

$$
R(t) = \frac{p(t) + D(t) - p(t-1)}{p(t-1)}
$$

---

## 2. 对数收益率（Log Return）

在量化研究和风险管理中，常使用**对数收益率**。其定义为：

$$
r(t) = \ln\left( \frac{p(t)}{p(t-1)} \right)
$$

特点：

* 对数收益率具有时间可加性（累乘变累加）：
  $$
  r(1) + r(2) + \cdots + r(T) = \ln\left( \frac{p(T)}{p(0)} \right)
  $$
* 当收益较小时，对数收益率近似等于简单收益率：
  $$
  r(t) \approx R(t)
  $$

---

In [2]:
# 先获取数据

df = ak.stock_zh_a_hist(
    symbol="000001",  #选取000001平安银行
    start_date="20000101",
    end_date="20251231",
    adjust="qfq"   # 使用前复权
)

df.info()
#数据进行保存
df.to_excel('历史行情数据.xlsx',index = False)
print('数据保存成功')
df.tail()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6112 entries, 0 to 6111
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   日期      6112 non-null   object 
 1   股票代码    6112 non-null   object 
 2   开盘      6112 non-null   float64
 3   收盘      6112 non-null   float64
 4   最高      6112 non-null   float64
 5   最低      6112 non-null   float64
 6   成交量     6112 non-null   int64  
 7   成交额     6112 non-null   float64
 8   振幅      6112 non-null   float64
 9   涨跌幅     6112 non-null   float64
 10  涨跌额     6112 non-null   float64
 11  换手率     6112 non-null   float64
dtypes: float64(9), int64(1), object(2)
memory usage: 573.1+ KB
数据保存成功


,日期,股票代码,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率
6107,2025-11-17,000001,11.75,11.67,11.75,11.62,995233,1.161416e+09,1.11,-0.68,-0.08,0.51
6108,2025-11-18,000001,11.65,11.59,11.69,11.57,670077,7.788876e+08,1.03,-0.69,-0.08,0.35
6109,2025-11-19,000001,11.58,11.80,11.83,11.57,1334996,1.565429e+09,2.24,1.81,0.21,0.69
6110,2025-11-20,000001,11.79,11.85,11.99,11.74,1637040,1.948416e+09,2.12,0.42,0.05,0.84
6111,2025-11-21,000001,11.80,11.69,11.88,11.66,1465359,1.721871e+09,1.86,-1.35,-0.16,0.76


In [3]:
# 先复制一份
df_r = df.copy()

# 过滤掉价格为 0 或缺失的行（避免 log 的问题）
df_r = df_r[df_r["收盘"] > 0].copy()

# 简单收益率：还是用 pct_change
df_r["simple_return"] = df_r["收盘"].pct_change()

# 取对数价格
df_r["log_price"] = np.log(df_r["收盘"])

# 对数收益率：直接对 log 价格做一阶差分
df_r["log_return"] = df_r["log_price"].diff(1)

# 一般可以把第一行 NaN 去掉
df_r = df_r.dropna(subset=["simple_return", "log_return"])

In [4]:
df_r.head()

,日期,股票代码,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率,simple_return,log_price,log_return
1,2000-01-05,000001,0.60,0.54,0.70,0.53,93993,173475008.0,28.81,-8.47,-0.05,0.88,-0.084746,-0.616186,-0.088553
2,2000-01-06,000001,0.53,0.68,0.73,0.48,120222,221192000.0,46.30,25.93,0.14,1.12,0.259259,-0.385662,0.230524
3,2000-01-07,000001,0.72,0.83,0.87,0.71,229346,443592000.0,23.53,22.06,0.15,2.14,0.220588,-0.186330,0.199333
4,2000-01-10,000001,0.88,0.95,1.01,0.87,185210,372294016.0,16.87,14.46,0.12,1.73,0.144578,-0.051293,0.135036
5,2000-01-11,000001,0.95,0.72,0.96,0.69,126663,245867008.0,28.42,-24.21,-0.23,1.18,-0.242105,-0.328504,-0.277211


### 我已经得到股票数据结构如下，请提取日期和收益率序列，其中日期重命名为date，改成日期格式，最后的数据按照时间建议索引。请写出代码。

In [5]:
# 只提取日期、简单收益、对数收益
df_return = df_r[["日期", "收盘", "simple_return", "log_return"]].copy()

# 重命名“日期”为 date
df_return.rename(columns={"日期": "date", "收盘": "price"}, inplace=True)

# 转换为 datetime 格式
df_return["date"] = pd.to_datetime(df_return["date"], errors="coerce")

# 按时间排序
df_return = df_return.sort_values("date")

# 设置日期为索引
df_return = df_return.set_index("date")

# 输出前几行检查
print(df_return.head())

            price  simple_return  log_return
date                                        
2000-01-05   0.54      -0.084746   -0.088553
2000-01-06   0.68       0.259259    0.230524
2000-01-07   0.83       0.220588    0.199333
2000-01-10   0.95       0.144578    0.135036
2000-01-11   0.72      -0.242105   -0.277211


### 3. 经通货膨胀调整的收益率（Real Return）

在金融分析中，名义收益率并不能真实反映投资者的**购买力变动**。
原因是：

> **如果物价上涨，资产名义收益的一部分只是用于抵消通胀，而不是真正让投资者变得更富有。**

因此我们需要计算**实际收益率（Real Return）**，它衡量的是投资在扣除通货膨胀之后的“真实回报”。

#### （1） 实际收益率

实际收益率表示：

> 在扣除当期物价上涨后的情况下，投资的购买力究竟提升了多少。

如果名义收益为 8%，通胀为 5%，实际收益不会是 8%，而会更低。

---

#### （2） 实际收益率计算公式

$$
R_{\text{real}}(t) = \frac{1+R(t)}{1+\pi(t)} - 1
$$

含义解释：

* $1+R(t)$：资产名义回报
* $1+\pi(t)$：物价上涨率（购买力下降率）
* 两者相除：得到购买力意义上的真实增幅
* 再减 1：得到实际收益率

In [6]:
# 导入通胀数据
cpi_raw = ak.macro_china_cpi_monthly()

cpi_raw.head()

,商品,日期,今值,预测值,前值
0,中国CPI月率报告,1996-02-01,2.1,NaN,NaN
1,中国CPI月率报告,1996-03-01,2.3,NaN,2.1
2,中国CPI月率报告,1996-04-01,0.6,NaN,2.3
3,中国CPI月率报告,1996-05-01,0.7,NaN,0.6
4,中国CPI月率报告,1996-06-01,-0.5,NaN,0.7


In [16]:
# 只保留两列：日期、今值
cpi = cpi_raw[["日期", "今值"]].copy()

# 改名
cpi.rename(columns={"日期": "date", "今值": "inf"}, inplace=True)

# 转换日期格式
cpi["date"] = pd.to_datetime(cpi["date"], errors="coerce")

# 通胀率从百分比转为小数
cpi["inflation"] = cpi["inf"] / 100.0

# 保证排序
cpi = cpi.sort_values("date")

print("\n>>> 清洗后的 CPI：")
print(cpi.head())


>>> 清洗后的 CPI：
        date  inf  inflation
0 1996-02-01  2.1      0.021
1 1996-03-01  2.3      0.023
2 1996-04-01  0.6      0.006
3 1996-05-01  0.7      0.007
4 1996-06-01 -0.5     -0.005


## 提示词：由于股票收益率序列是日度数据，通胀是月度数据，因此请先将股票数据重采样后再与通胀序列按照时间索引合并

### 请介绍重采样的原理，一共分三个形式：  
1.按照股票价格该月最后一个交易日的价格采样  
2.采用简单收益率计算  
3.采用对数收益率计算  
同时请给出相应的代码，每个代码写上注释，分模块给出。

### 形式一：按该月最后一个交易日的价格采样

#### 原理说明

设某资产在时点 $t$ 的价格为 $p(t)$。
对于某一月份（比如 2020 年 1 月），假设当月最后一个交易日为 $T_{\text{Jan}}$，则：

 **该月价格代表值**可以取为
 $$
 p_{\text{month}} = p(T_{\text{Jan}})
 $$

在实际应用中，这种做法常用于构造“月度价格序列”，后续可再基于月度价格计算收益率。

In [8]:
# ================== 模块 A：按月末价格重采样 ==================

# 1）按月重采样，取该月最后一个交易日的收盘价
monthly_price = df_return["price"].resample("M").last()

# 2）可选：如果需要变成 DataFrame，并重命名列
monthly_price = monthly_price.to_frame(name="close_month_end")

print(monthly_price.head())

            close_month_end
date                       
2000-01-31             0.63
2000-02-29             0.59
2000-03-31             0.60
2000-04-30             0.73
2000-05-31             0.53


In [9]:
# ================== 模块 A：月末价格采样 ==================

# 1. 取月末收盘价
monthly_price = df_return["price"].resample("M").last()

# 2. 根据月末价格计算月度简单收益率
monthly_return_from_price = monthly_price.pct_change()

monthly_return_from_price = monthly_return_from_price.to_frame(name="return_from_price")

print(monthly_return_from_price.head())

            return_from_price
date                         
2000-01-31                NaN
2000-02-29          -0.063492
2000-03-31           0.016949
2000-04-30           0.216667
2000-05-31          -0.273973


### 形式二：基于简单收益率的月度收益率 
#### 原理说明 
 设第 $t$ 天的**简单收益率**为 $R_d(t)$，即 
 $$ 
 R_d(t) = \frac{p(t) - p(t-1)}{p(t-1)} 
 $$ 
 在一个月内，若该月的交易日集合记为 $\mathcal{D}$，则**该月的总简单收益率**为多期复利： 
 $$ 
 R_m = \prod_{t \in \mathcal{D}} \big(1 + R_d(t)\big) - 1 
 $$ 
 也就是把**每天的 $(1+R_d)$ 连乘，再减 1**。

In [10]:
# ================== 模块 B：简单收益率 → 月度简单收益率 ==================

# 1）计算每个月的复利收益率：
#    R_m = (1 + R_d1) * (1 + R_d2) * ... * (1 + R_dn) - 1
monthly_simple_return = (1 + df_return["simple_return"]).resample("M").prod() - 1

# 2）整理成 DataFrame
monthly_simple_return = monthly_simple_return.to_frame(name="simple_return_m")

print(monthly_simple_return.head())

            simple_return_m
date                       
2000-01-31         0.067797
2000-02-29        -0.063492
2000-03-31         0.016949
2000-04-30         0.216667
2000-05-31        -0.273973


### 形式三：基于对数收益率的月度收益率 
#### 原理说明 
 对数收益率定义为： 
 $$ 
 r_d(t) = \ln\left(\frac{p(t)}{p(t-1)}\right) 
 $$ 
 由于对数收益率具有**可加性**： 
 $$ 
 r_m = \sum_{t \in \mathcal{D}} r_d(t) 
 $$ 
 也就是说，**月度对数收益率就是当月所有日度对数收益率的和**。 
 如果需要从对数收益率回到简单收益率，可以用： 
 $$ 
 R_m = e^{r_m} - 1 
 $$

In [11]:
# ================== 模块 C：对数收益率 → 月度对数收益率 ==================

# 1）月度对数收益率：按月对日度对数收益率求和
monthly_log_return = df_return["log_return"].resample("M").sum()

# 2）整理成 DataFrame
monthly_log_return = monthly_log_return.to_frame(name="log_return_m")

print(monthly_log_return.head())

            log_return_m
date                    
2000-01-31      0.065597
2000-02-29     -0.065597
2000-03-31      0.016807
2000-04-30      0.196115
2000-05-31     -0.320168


In [12]:
# 合并到一个 DataFrame 中 
monthly_all = monthly_return_from_price.join([monthly_simple_return, monthly_log_return], how="inner") 
monthly_all = monthly_all.dropna()
print(monthly_all.head())

## 从结果可以看到符合预期

            return_from_price  simple_return_m  log_return_m
date                                                        
2000-02-29          -0.063492        -0.063492     -0.065597
2000-03-31           0.016949         0.016949      0.016807
2000-04-30           0.216667         0.216667      0.196115
2000-05-31          -0.273973        -0.273973     -0.320168
2000-06-30           0.056604         0.056604      0.055060


### 提示词：我现在有一个叫monthly_all的dataframe,分别是三种不同方式计算的股票收益率，还有一个是cpi的列表数据，请将他们按照日期合并，再计算经过通过膨胀调整的股票收益率。（可以贴上数据）
###将：  
     •  monthly_all（月度收益率，索引=月末日期）    
     •  cpi（月初日期 + inflation 列）     
      进行合并，并计算 实际收益率（real return）。   
      重点是： CPI 是“月初日期（如 2000-02-01）”，monthly_all 是“月末日期（如 2000-02-29）”。  要对齐，只需把 CPI 的日期转换为当月月末，然后按 index 合并即可。

In [17]:
# ======================
# 1. 处理 CPI 数据（将月初 → 月末）
# ======================
# 将 CPI 日期转换为该月的“月末日期”
cpi["date"] = cpi["date"].dt.to_period("M").dt.to_timestamp("M")

# 设为索引
cpi = cpi.set_index("date").sort_index()

# 只保留 inflation 列
cpi_m = cpi[["inflation"]]

print(cpi_m)

            inflation
date                 
1996-02-29      0.021
1996-03-31      0.023
1996-04-30      0.006
1996-05-31      0.007
1996-06-30     -0.005
...               ...
2025-05-31      0.001
2025-06-30     -0.002
2025-07-31     -0.001
2025-08-31      0.004
2025-09-30        NaN

[357 rows x 1 columns]


In [18]:
# ======================
# 2. 直接按日期 index 合并
# ======================
merged = monthly_all.join(cpi_m, how="left")

print("\n合并后的数据：")
print(merged.head())


合并后的数据：
            return_from_price  simple_return_m  log_return_m  inflation
date                                                                   
2000-02-29          -0.063492        -0.063492     -0.065597      0.009
2000-03-31           0.016949         0.016949      0.016807      0.019
2000-04-30           0.216667         0.216667      0.196115     -0.016
2000-05-31          -0.273973        -0.273973     -0.320168     -0.009
2000-06-30           0.056604         0.056604      0.055060     -0.010


In [19]:
# ======================
# 3. 计算实际收益率（Real Return）
#    公式：R_real = (1 + R_nominal) / (1 + inflation) - 1
# ======================

# 简单收益率调整后的实际收益率
merged["real_return_simple"] = (1 + merged["simple_return_m"]) / (1 + merged["inflation"]) - 1

# 月末价格方式计算的实际收益率
merged["real_return_from_price"] = (1 + merged["return_from_price"]) / (1 + merged["inflation"]) - 1

# 对数收益率方式（需先转为简单收益率）
R_from_log = np.exp(merged["log_return_m"]) - 1
merged["real_return_from_log"] = (1 + R_from_log) / (1 + merged["inflation"]) - 1


print("\n最终含实际收益率的数据：")
print(merged.head())


最终含实际收益率的数据：
            return_from_price  simple_return_m  log_return_m  inflation  \
date                                                                      
2000-02-29          -0.063492        -0.063492     -0.065597      0.009   
2000-03-31           0.016949         0.016949      0.016807      0.019   
2000-04-30           0.216667         0.216667      0.196115     -0.016   
2000-05-31          -0.273973        -0.273973     -0.320168     -0.009   
2000-06-30           0.056604         0.056604      0.055060     -0.010   

            real_return_simple  real_return_from_price  real_return_from_log  
date                                                                          
2000-02-29           -0.071845               -0.071845             -0.071845  
2000-03-31           -0.002013               -0.002013             -0.002013  
2000-04-30            0.236450                0.236450              0.236450  
2000-05-31           -0.267379               -0.267379           

## 计算波动率
### 📌 **1. 原理：什么是波动率？**

在金融资产（股票、期货、债券等）的时间序列中：

* **波动率（Volatility）**衡量的是资产收益率随时间波动的幅度
* 可以理解为价格变化的“平均幅度”
* 波动率越大，资产风险越高

一般使用 **收益率的标准差（Standard Deviation）** 作为波动率的衡量指标。

> **核心思想：波动率 = 收益率“离散程度”的度量**

---

### 📌 **2. 波动率计算步骤（按你要求分五步）**

下面以“日度价格 → 日度波动率”为例。

---

#### **步骤 1：选择波动率计算的时间段**

例如：

* 过去 20 天（日度滚动波动率）
* 一个季度、半年、一年
* 或者月度收益率序列

时间段越长，估计越平稳。

---

#### **步骤 2：计算价格变化（收益率）**

使用简单收益率或对数收益率均可。

常用的是对数收益率：

$$
r_t = \ln \left(\frac{p_t}{p_{t-1}}\right)
$$

原因：对数收益可加性强，适合累计分析。

---

#### **步骤 3：计算收益率的平方**

每一天的收益率平方：

$$
r_t^2
$$

平方后，能让“涨跌方向不互相抵消”，只度量“幅度”。

---

#### **步骤 4：求和 / 求平均**

对该时间段内的收益率平方求平均：

$$
\frac{1}{n-1}\sum_{t=1}^n \left(r_t - \bar{r}\right)^2
$$

若假设收益率均值≈0（高频金融数据常见假设）：

$$
\frac{1}{n-1}\sum_{t=1}^n r_t^2
$$

---

#### **步骤 5：取平方根（得到标准差 = 波动率）**

波动率是方差的平方根：

$$
\sigma = \sqrt{\frac{1}{n-1}\sum_{t=1}^n (r_t - \bar{r})^2 }
$$

如果假设均值为 0：

$$
\sigma = \sqrt{\frac{1}{n-1}\sum_{t=1}^n r_t^2 }
$$

---

In [20]:
# 按照方差公式手工计算波动率（基于 log_return）
r = df_return["log_return"].dropna()

# 均值
r_mean = r.mean()

# 平方偏差
sq = (r - r_mean) ** 2

# 样本方差
variance = sq.sum() / (len(r) - 1)

# 标准差 = 波动率
vol_manual = variance ** 0.5

print("手工计算波动率：", vol_manual)

手工计算波动率： 0.1357040319299861


### 月度实现波动率（Realized Volatility）的计算原理

#### 1. 原理

月度实现波动率是用**该月内所有日度收益率的波动程度**来度量当月风险大小的一种指标。  
常见做法是：

> 把当月所有日度收益率先平方，再求和，最后取平方根。

- 使用**平方**是为了消除正负号，只度量“波动幅度”
- 对平方**求和**是把整个月的信息累积起来
- 最后**开平方根**，把量纲从“平方收益率”还原为“收益率”的量级

这就是“基于收益率平方和”的**实现波动率**思想。

---

#### 2. 计算步骤（以日度收益率构造月度实现波动率）

设某一月份内的交易日集合为该月包含的所有日 $t=1,\dots,N$。

1. **选择时间段**  
   确定要计算的月份（例如 2000 年 2 月），找出该月所有交易日。

2. **计算日度收益率**  
   对每个交易日，用价格计算日度收益率（通常用对数收益率）：
   - 简单收益率：$R_t = \dfrac{p_t - p_{t-1}}{p_{t-1}}$  
   - 或对数收益率：$r_t = \ln\big( \dfrac{p_t}{p_{t-1}} \big)$

3. **计算收益率平方**  
   对每一天的收益率求平方：
   - 若用对数收益率：$r_t^2$
   - 若用简单收益率：$R_t^2$

4. **对当月所有平方项求和**  
   将该月内所有日度收益率的平方加总：
   - $\sum\_{t=1}^{N} r_t^2$  或  $\sum\_{t=1}^{N} R_t^2$

5. **对平方和取平方根**  
   用平方根把“平方收益率”的量纲还原为“收益率”的量级，得到该月的实现波动率：
   - $\sqrt{\sum\_{t=1}^{N} r_t^2}$

---

#### 3. 数学公式

1. **日度对数收益率**

$$
r_t = \ln\left(\frac{p_t}{p_{t-1}}\right)
$$

2. **某一月份的实现波动率（基于日度收益率平方和）**

设该月共有 $N$ 个交易日，则月度实现波动率为：

$$
RV_m = \sqrt{\sum_{t=1}^{N} r_t^{2}}
$$

In [21]:
# 确保 index 是 DatetimeIndex
df_return = df_return.sort_index()

# 1) 计算收益率平方
df_return["r_sq"] = df_return["log_return"] ** 2

# 2) 按月对收益率平方求和
monthly_rsq_sum = df_return["r_sq"].resample("M").sum()

# 3) 取平方根 → 实现波动率
monthly_realized_vol = np.sqrt(monthly_rsq_sum)

# 整理成 DataFrame
monthly_realized_vol = monthly_realized_vol.to_frame(name="realized_vol")

print(monthly_realized_vol.head())

            realized_vol
date                    
2000-01-31      0.627321
2000-02-29      0.619965
2000-03-31      0.616184
2000-04-30      0.299469
2000-05-31      0.390291


In [25]:
#  年化波动率
vol_annual = monthly_realized_vol * np.sqrt(12)
print("年化波动率：", vol_annual)

年化波动率：             realized_vol
date                    
2000-01-31      2.173102
2000-02-29      2.147622
2000-03-31      2.134523
2000-04-30      1.037391
2000-05-31      1.352008
...                  ...
2025-07-31      0.221611
2025-08-31      0.167488
2025-09-30      0.137410
2025-10-31      0.104424
2025-11-30      0.111534

[311 rows x 1 columns]


### 以下是缺失值的处理，以cpi为例

In [27]:
# 查看每列的缺失数量
print(cpi.isna().sum())

inf          3
inflation    3
dtype: int64


In [28]:
# 查看日期的缺失情况
date_missing = cpi_raw["日期"].isna().sum()
print("日期缺失数量：", date_missing)

日期缺失数量： 0


In [34]:
missing_rows = cpi[cpi["inflation"].isna()][["inf", "inflation"]]

print("缺失的 inflation 日期：")
print(missing_rows)

缺失的 inflation 日期：
            inf  inflation
date                      
2019-11-30  NaN        NaN
2025-01-31  NaN        NaN
2025-09-30  NaN        NaN


In [35]:
# 用向前填充的方式
cpi_ffill = cpi.sort_index().copy()

# 向前填充 inflation
cpi_ffill["inflation"] = cpi_ffill["inflation"].fillna(method="ffill")

print("向前填充后的头几行：")
print(cpi_ffill.head())

向前填充后的头几行：
            inf  inflation
date                      
1996-02-29  2.1      0.021
1996-03-31  2.3      0.023
1996-04-30  0.6      0.006
1996-05-31  0.7      0.007
1996-06-30 -0.5     -0.005


In [36]:
# 用线性插值的方式进行填充
cpi_interp = cpi.sort_index().copy()

# 线性插值
cpi_interp["inflation"] = cpi_interp["inflation"].interpolate(method="linear")

print("线性插值后的结果：")
print(cpi_interp.head())

线性插值后的结果：
            inf  inflation
date                      
1996-02-29  2.1      0.021
1996-03-31  2.3      0.023
1996-04-30  0.6      0.006
1996-05-31  0.7      0.007
1996-06-30 -0.5     -0.005
